# PhyGrainNet — Kaggle runner

Attach the competition data (**Add Input → Competitions → soil-grain-size-from-photos**), turn on **Internet** (needed for `git clone`) and pick **GPU T4 x2** for the neural cells.
Run the sections you need for the current week (see `docs/SCHEDULE.md`). Every run writes to `/kaggle/working/outputs/<experiment_id>/` (oof.csv, test.csv, metrics.json) and appends a row to `experiments/registry.csv`.

In [ ]:
BRANCH = "main"   # or a feature branch
!rm -rf /kaggle/working/PhyGrainNet
!git clone -q -b $BRANCH https://github.com/rexxar280903/PhyGrainNet.git /kaggle/working/PhyGrainNet
%cd /kaggle/working/PhyGrainNet
!pip install -q -e . 2>&1 | tail -1
!git log --oneline -1

## 1. Audit (P2) — run once, copy the summary into docs/DATASET.md

In [ ]:
!python scripts/audit_dataset.py --data-root /kaggle/input/soil-grain-size-from-photos

## 2. A000 — first valid submission (training median) + image-free CV

In [ ]:
!python scripts/baseline_prior.py --out /kaggle/working/submission_A000_median.csv

## 3. A001 — physically scaled features + small models (CPU is enough)

In [ ]:
!python scripts/train.py --config configs/classical/features_v1.yaml

## 4. C100 — PhyGrainNet-MV from random init (GPU)
For a quick check first: append `training.epochs=2 cv.max_folds=1`.

In [ ]:
!python scripts/train.py --config configs/phygrainnet/mv_scratch.yaml

## 5. P000 → D100 — own SimCLR pretraining, then fine-tune

In [ ]:
!python scripts/train.py --config configs/pretrain/ssl.yaml
!python scripts/train.py --config configs/phygrainnet/mv_ssl.yaml

## 6. P100 → D200 — ETS supervised pretraining (needs the ETS Kaggle dataset as input)

In [ ]:
# !python scripts/convert_ets_labels.py --input /kaggle/input/ets-photogranulometry-10ppm/psd.xlsx --output /kaggle/working/ets_labels_11pt.csv
!python scripts/train.py --config configs/pretrain/ets.yaml
!python scripts/train.py --config configs/phygrainnet/mv_ets.yaml

## 7. Ensemble (pointwise median, greedy on OOF) → validated submission

In [ ]:
import glob
runs = sorted(glob.glob('/kaggle/working/outputs/*/oof.csv'))
print('\n'.join(runs))

In [ ]:
!python scripts/ensemble.py --runs /kaggle/working/outputs/A001_ridge10 /kaggle/working/outputs/C100 --out /kaggle/working/submission_H001.csv
!python scripts/validate.py /kaggle/working/submission_H001.csv

## 8. Leaderboard for the registry
Copy new rows of `experiments/registry.csv` into the GitHub repo (commit `exp(<id>): ...`).

In [ ]:
import pandas as pd
pd.read_csv('experiments/registry.csv').sort_values('cv_emd').head(20)